In [1]:
# ============================================================
# FINAL DATASET AUDIT
# Cell 1 — Imports and configuration
# ============================================================

from pathlib import Path
import sys
import json
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\Pancreatic_Cancer_Thesis")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

METADATA_PATH = PROCESSED_DIR / "metadata.csv"

IMAGES_DIR = PROCESSED_DIR / "images"
MASKS_DIR = PROCESSED_DIR / "masks"

print("=" * 70)
print("FINAL DATASET AUDIT")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nMetadata:")
print(METADATA_PATH)

print("\nImages:")
print(IMAGES_DIR)

print("\nMasks:")
print(MASKS_DIR)

print("\nExistence:")
print("  metadata.csv :", METADATA_PATH.exists())
print("  images/      :", IMAGES_DIR.exists())
print("  masks/       :", MASKS_DIR.exists())

FINAL DATASET AUDIT

Project root:
D:\Pancreatic_Cancer_Thesis

Metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

Images:
D:\Pancreatic_Cancer_Thesis\data\processed\images

Masks:
D:\Pancreatic_Cancer_Thesis\data\processed\masks

Existence:
  metadata.csv : True
  images/      : True
  masks/       : True


In [2]:
# ============================================================
# Cell 2 — Load processed metadata
# ============================================================

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"metadata.csv not found:\n{METADATA_PATH}"
    )

metadata = pd.read_csv(METADATA_PATH)

print("=" * 70)
print("METADATA OVERVIEW")
print("=" * 70)

print("Rows    :", len(metadata))
print("Columns :", len(metadata.columns))

print("\nColumns:")
for col in metadata.columns:
    print(" ", col)

print("\nFirst 5 rows:")
display(metadata.head())

METADATA OVERVIEW
Rows    : 2238
Columns : 18

Columns:
  study_id
  label
  image_path
  mask_path
  image_shape
  mask_shape
  image_dtype
  mask_dtype
  mask_labels
  patient_id
  patient_age
  patient_sex
  scanner
  diagnosis
  diagnosis_source
  roi_size
  target_spacing
  hu_window

First 5 rows:


,study_id,label,image_path,mask_path,image_shape,mask_shape,image_dtype,mask_dtype,mask_labels,patient_id,patient_age,patient_sex,scanner,diagnosis,diagnosis_source,roi_size,target_spacing,hu_window
0,100000_00001,NaN,images\100000_00001.npy,masks\100000_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,6",100000,42.0,F,TOSHIBA,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
1,100001_00001,NaN,images\100001_00001.npy,masks\100001_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100001,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
2,100002_00001,NaN,images\100002_00001.npy,masks\100002_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100002,77.0,F,TOSHIBA,PDAC,pathology,"128,160,192","1.0,1.0,3.0","-150,250"
3,100003_00001,NaN,images\100003_00001.npy,masks\100003_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100003,57.0,M,TOSHIBA,PDAC,cytology,"128,160,192","1.0,1.0,3.0","-150,250"
4,100004_00001,NaN,images\100004_00001.npy,masks\100004_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100004,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"


In [3]:
# ============================================================
# Cell 3 — Identify important metadata columns
# ============================================================

print("=" * 70)
print("IDENTIFYING IMPORTANT METADATA COLUMNS")
print("=" * 70)

def find_column(df, candidates):
    """
    Find the first matching column from a list of candidates.
    Matching is case-insensitive.
    """
    lookup = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


STUDY_ID_COL = find_column(
    metadata,
    ["study_id", "studyid", "case_id", "caseid"]
)

PATIENT_ID_COL = find_column(
    metadata,
    ["patient_id", "patientid", "subject_id", "subjectid"]
)

DIAGNOSIS_COL = find_column(
    metadata,
    ["diagnosis", "label", "class", "target"]
)

BATCH_COL = find_column(
    metadata,
    ["batch", "audit_batch", "source_batch"]
)

LABEL_TYPE_COL = find_column(
    metadata,
    ["label_type", "selected_label_type"]
)

print("Study ID column    :", STUDY_ID_COL)
print("Patient ID column  :", PATIENT_ID_COL)
print("Diagnosis column   :", DIAGNOSIS_COL)
print("Batch column       :", BATCH_COL)
print("Label type column  :", LABEL_TYPE_COL)

required = {
    "study_id": STUDY_ID_COL,
    "patient_id": PATIENT_ID_COL,
    "diagnosis": DIAGNOSIS_COL,
}

missing = [
    name for name, value in required.items()
    if value is None
]

if missing:
    print("\n⚠ Missing expected columns:", missing)
    print("Available columns:", list(metadata.columns))
else:
    print("\n✓ Required audit columns identified.")

IDENTIFYING IMPORTANT METADATA COLUMNS
Study ID column    : study_id
Patient ID column  : patient_id
Diagnosis column   : diagnosis
Batch column       : None
Label type column  : None

✓ Required audit columns identified.


In [4]:
# ============================================================
# Cell 4 — Dataset composition
# ============================================================

print("=" * 70)
print("FINAL DATASET COMPOSITION")
print("=" * 70)

print("\nTotal cases:", len(metadata))

if DIAGNOSIS_COL is not None:
    print("\nDiagnosis distribution:")
    print(metadata[DIAGNOSIS_COL].value_counts(dropna=False))

    print("\nDiagnosis percentages:")
    print(
        metadata[DIAGNOSIS_COL]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("\nCross-tabulation:")
    display(
        pd.crosstab(
            metadata[DIAGNOSIS_COL],
            columns="count"
        )
    )

FINAL DATASET COMPOSITION

Total cases: 2238

Diagnosis distribution:
diagnosis
non-PDAC    1562
PDAC         676
Name: count, dtype: int64

Diagnosis percentages:
diagnosis
non-PDAC    69.79
PDAC        30.21
Name: proportion, dtype: float64

Cross-tabulation:


col_0,count
diagnosis,
PDAC,676
non-PDAC,1562


In [5]:
# ============================================================
# Cell 5 — Expected final dataset count
# ============================================================

EXPECTED_CASES = 2238
EXPECTED_PDAC = 676
EXPECTED_NON_PDAC = 1562

print("=" * 70)
print("EXPECTED DATASET COUNT CHECK")
print("=" * 70)

actual_cases = len(metadata)

print("Expected total cases :", EXPECTED_CASES)
print("Actual total cases   :", actual_cases)

if actual_cases == EXPECTED_CASES:
    print("✓ Total case count matches expected 2,238.")
else:
    print("✗ WARNING: Total case count does not match!")

if DIAGNOSIS_COL is not None:
    diagnosis_counts = metadata[DIAGNOSIS_COL].value_counts()

    pdac_count = int(diagnosis_counts.get("PDAC", 0))
    non_pdac_count = int(diagnosis_counts.get("non-PDAC", 0))

    print("\nPDAC:")
    print("  Expected:", EXPECTED_PDAC)
    print("  Actual  :", pdac_count)

    print("\nNon-PDAC:")
    print("  Expected:", EXPECTED_NON_PDAC)
    print("  Actual  :", non_pdac_count)

    if pdac_count == EXPECTED_PDAC:
        print("✓ PDAC count matches.")
    else:
        print("✗ WARNING: PDAC count mismatch.")

    if non_pdac_count == EXPECTED_NON_PDAC:
        print("✓ Non-PDAC count matches.")
    else:
        print("✗ WARNING: Non-PDAC count mismatch.")

EXPECTED DATASET COUNT CHECK
Expected total cases : 2238
Actual total cases   : 2238
✓ Total case count matches expected 2,238.

PDAC:
  Expected: 676
  Actual  : 676

Non-PDAC:
  Expected: 1562
  Actual  : 1562
✓ PDAC count matches.
✓ Non-PDAC count matches.


In [6]:
# ============================================================
# Cell 6 — Study ID uniqueness
# ============================================================

print("=" * 70)
print("STUDY ID UNIQUENESS")
print("=" * 70)

if STUDY_ID_COL is None:
    print("⚠ Study ID column not found.")
else:
    total = len(metadata)
    unique = metadata[STUDY_ID_COL].nunique(dropna=False)
    duplicates = total - unique

    print("Total study IDs :", total)
    print("Unique study IDs:", unique)
    print("Duplicate IDs   :", duplicates)

    if duplicates == 0:
        print("\n✓ Every processed case has a unique study ID.")
    else:
        print("\n✗ WARNING: Duplicate study IDs detected.")

        dup_ids = (
            metadata.loc[
                metadata[STUDY_ID_COL].duplicated(keep=False),
                STUDY_ID_COL
            ]
            .sort_values()
            .unique()
        )

        print("\nDuplicate study IDs:")
        for sid in dup_ids:
            print(" ", sid)

STUDY ID UNIQUENESS
Total study IDs : 2238
Unique study IDs: 2238
Duplicate IDs   : 0

✓ Every processed case has a unique study ID.


In [7]:
# ============================================================
# Cell 7 — Patient-level duplication / leakage audit
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL DUPLICATION AUDIT")
print("=" * 70)

if PATIENT_ID_COL is None:
    raise KeyError(
        "Patient ID column could not be identified. "
        "Patient-level splitting cannot proceed safely."
    )

total_cases = len(metadata)
unique_patients = metadata[PATIENT_ID_COL].nunique(dropna=False)

print("Total cases     :", total_cases)
print("Unique patients :", unique_patients)
print("Extra cases     :", total_cases - unique_patients)

patient_case_counts = (
    metadata
    .groupby(PATIENT_ID_COL)
    .size()
    .sort_values(ascending=False)
)

multi_case_patients = patient_case_counts[
    patient_case_counts > 1
]

print("\nPatients with multiple cases:",
      len(multi_case_patients))

print("\nDistribution of cases per patient:")
print(patient_case_counts.value_counts().sort_index())

if len(multi_case_patients) > 0:
    print("\nPatients with multiple cases:")
    display(
        multi_case_patients
        .rename("case_count")
        .to_frame()
    )

PATIENT-LEVEL DUPLICATION AUDIT
Total cases     : 2238
Unique patients : 2224
Extra cases     : 14

Patients with multiple cases: 11

Distribution of cases per patient:
1    2213
2      10
5       1
Name: count, dtype: int64

Patients with multiple cases:


,case_count
patient_id,
100047,5
100265,2
100455,2
100416,2
100268,2
100417,2
100456,2
101361,2
101595,2


In [8]:
# ============================================================
# Cell 8 — Patient-level diagnosis consistency
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL DIAGNOSIS CONSISTENCY")
print("=" * 70)

if DIAGNOSIS_COL is None:
    print("⚠ Diagnosis column not found.")
else:
    patient_diagnosis = (
        metadata
        .groupby(PATIENT_ID_COL)[DIAGNOSIS_COL]
        .nunique(dropna=False)
    )

    inconsistent_patients = patient_diagnosis[
        patient_diagnosis > 1
    ]

    print(
        "Patients with multiple diagnosis values:",
        len(inconsistent_patients)
    )

    if len(inconsistent_patients) == 0:
        print("\n✓ No patient has conflicting diagnosis labels.")
    else:
        print(
            "\n⚠ WARNING: Patients with conflicting diagnoses:"
        )

        display(
            metadata[
                metadata[PATIENT_ID_COL].isin(
                    inconsistent_patients.index
                )
            ]
            .sort_values(PATIENT_ID_COL)
        )

PATIENT-LEVEL DIAGNOSIS CONSISTENCY
Patients with multiple diagnosis values: 0

✓ No patient has conflicting diagnosis labels.


In [9]:
# ============================================================
# Cell 9 — Diagnosis distribution at patient level
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL DIAGNOSIS DISTRIBUTION")
print("=" * 70)

patient_level = (
    metadata
    .drop_duplicates(subset=[PATIENT_ID_COL])
    [[PATIENT_ID_COL, DIAGNOSIS_COL]]
)

print("Unique patients:", len(patient_level))

print("\nPatients by diagnosis:")
print(
    patient_level[DIAGNOSIS_COL]
    .value_counts(dropna=False)
)

print("\nPatient diagnosis percentages:")
print(
    patient_level[DIAGNOSIS_COL]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

PATIENT-LEVEL DIAGNOSIS DISTRIBUTION
Unique patients: 2224

Patients by diagnosis:
diagnosis
non-PDAC    1551
PDAC         673
Name: count, dtype: int64

Patient diagnosis percentages:
diagnosis
non-PDAC    69.74
PDAC        30.26
Name: proportion, dtype: float64


In [10]:
# ============================================================
# Cell 10 — Batch × diagnosis audit
# ============================================================

print("=" * 70)
print("BATCH × DIAGNOSIS AUDIT")
print("=" * 70)

if BATCH_COL is None:
    print("⚠ Batch column not found.")
else:
    batch_diagnosis = pd.crosstab(
        metadata[BATCH_COL],
        metadata[DIAGNOSIS_COL],
        margins=True
    )

    display(batch_diagnosis)

    print("\nRow percentages:")
    display(
        pd.crosstab(
            metadata[BATCH_COL],
            metadata[DIAGNOSIS_COL],
            normalize="index"
        )
        .mul(100)
        .round(2)
    )

BATCH × DIAGNOSIS AUDIT
⚠ Batch column not found.


In [11]:
# ============================================================
# Cell 11 — Label type × diagnosis
# ============================================================

print("=" * 70)
print("LABEL TYPE × DIAGNOSIS")
print("=" * 70)

if LABEL_TYPE_COL is None:
    print("⚠ Label type column not found.")
else:
    table = pd.crosstab(
        metadata[LABEL_TYPE_COL],
        metadata[DIAGNOSIS_COL],
        margins=True
    )

    display(table)

    print("\nPercentages within diagnosis:")
    display(
        pd.crosstab(
            metadata[LABEL_TYPE_COL],
            metadata[DIAGNOSIS_COL],
            normalize="columns"
        )
        .mul(100)
        .round(2)
    )

LABEL TYPE × DIAGNOSIS
⚠ Label type column not found.


In [12]:
# ============================================================
# Cell 12 — Processed file presence audit
# ============================================================

print("=" * 70)
print("PROCESSED IMAGE / MASK PRESENCE AUDIT")
print("=" * 70)

if STUDY_ID_COL is None:
    raise KeyError("Study ID column required.")

study_ids = (
    metadata[STUDY_ID_COL]
    .astype(str)
    .tolist()
)

missing_images = []
missing_masks = []

for study_id in study_ids:

    image_path = IMAGES_DIR / f"{study_id}.npy"
    mask_path = MASKS_DIR / f"{study_id}.npy"

    if not image_path.exists():
        missing_images.append(study_id)

    if not mask_path.exists():
        missing_masks.append(study_id)

print("Metadata cases :", len(study_ids))
print("Missing images :", len(missing_images))
print("Missing masks  :", len(missing_masks))

if not missing_images:
    print("✓ All metadata cases have processed images.")
else:
    print("\nMissing images:")
    print(missing_images[:20])

if not missing_masks:
    print("✓ All metadata cases have processed masks.")
else:
    print("\nMissing masks:")
    print(missing_masks[:20])

PROCESSED IMAGE / MASK PRESENCE AUDIT
Metadata cases : 2238
Missing images : 0
Missing masks  : 0
✓ All metadata cases have processed images.
✓ All metadata cases have processed masks.


In [13]:
# ============================================================
# Cell 13 — Sample processed volume inspection
# ============================================================

print("=" * 70)
print("SAMPLE PROCESSED VOLUME INSPECTION")
print("=" * 70)

SAMPLE_SIZE = 10

sample_ids = sorted(study_ids)[:SAMPLE_SIZE]

results = []

for study_id in sample_ids:

    image_path = IMAGES_DIR / f"{study_id}.npy"
    mask_path = MASKS_DIR / f"{study_id}.npy"

    image = np.load(image_path, mmap_mode="r")
    mask = np.load(mask_path, mmap_mode="r")

    results.append({
        "study_id": study_id,
        "image_shape": image.shape,
        "image_dtype": str(image.dtype),
        "image_min": float(image.min()),
        "image_max": float(image.max()),
        "mask_shape": mask.shape,
        "mask_dtype": str(mask.dtype),
        "mask_min": int(mask.min()),
        "mask_max": int(mask.max()),
        "mask_labels": np.unique(mask).tolist(),
    })

sample_df = pd.DataFrame(results)

display(sample_df)

SAMPLE PROCESSED VOLUME INSPECTION


,study_id,image_shape,image_dtype,image_min,image_max,mask_shape,mask_dtype,mask_min,mask_max,mask_labels
0,100000_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 6]"
1,100001_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 5, 6]"
2,100002_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 1, 2, 3, 4, 5, 6]"
3,100003_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 1, 2, 3, 4, 5, 6]"
4,100004_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 5, 6]"
5,100005_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 1, 2, 3, 4, 6]"
6,100006_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 5, 6]"
7,100007_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 5, 6]"
8,100008_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 2, 3, 4, 5, 6]"
9,100009_00001,"(128, 160, 192)",float32,0.0,1.0,"(128, 160, 192)",uint8,0,6,"[0, 1, 2, 3, 4, 5, 6]"


In [14]:
# ============================================================
# Cell 14 — Mask label presence across the dataset
# ============================================================

print("=" * 70)
print("MASK LABEL PRESENCE")
print("=" * 70)

label_presence = {
    label: []
    for label in range(7)
}

for study_id in study_ids:

    mask_path = MASKS_DIR / f"{study_id}.npy"

    mask = np.load(
        mask_path,
        mmap_mode="r"
    )

    labels = np.unique(mask)

    for label in range(7):
        label_presence[label].append(
            label in labels
        )

presence_df = pd.DataFrame(
    label_presence,
    index=study_ids
)

presence_summary = pd.DataFrame({
    "label": range(7),
    "cases_with_label": [
        int(presence_df[label].sum())
        for label in range(7)
    ],
})

presence_summary["percentage"] = (
    presence_summary["cases_with_label"]
    / len(study_ids)
    * 100
).round(2)

display(presence_summary)

MASK LABEL PRESENCE


,label,cases_with_label,percentage
0,0,2238,100.00
1,1,676,30.21
2,2,2238,100.00
3,3,2238,100.00
4,4,2238,100.00
5,5,1778,79.45
6,6,2022,90.35


In [15]:
# ============================================================
# Cell 15 — PDAC lesion presence audit
# ============================================================

print("=" * 70)
print("PDAC LESION PRESENCE AUDIT")
print("=" * 70)

LESION_LABEL = 1

lesion_presence = []

for study_id in study_ids:

    mask_path = MASKS_DIR / f"{study_id}.npy"

    mask = np.load(
        mask_path,
        mmap_mode="r"
    )

    has_lesion = bool(np.any(mask == LESION_LABEL))

    lesion_presence.append({
        "study_id": study_id,
        "has_lesion": has_lesion
    })

lesion_df = pd.DataFrame(lesion_presence)

merged = metadata[
    [STUDY_ID_COL, DIAGNOSIS_COL]
].merge(
    lesion_df,
    left_on=STUDY_ID_COL,
    right_on="study_id",
    how="left"
)

print("\nLesion presence × diagnosis:")

display(
    pd.crosstab(
        merged[DIAGNOSIS_COL],
        merged["has_lesion"],
        margins=True
    )
)

PDAC LESION PRESENCE AUDIT

Lesion presence × diagnosis:


has_lesion,False,True,All
diagnosis,,,
PDAC,0,676,676
non-PDAC,1562,0,1562
All,1562,676,2238


In [16]:
# ============================================================
# Cell 16 — Final leakage / integrity summary
# ============================================================

print("=" * 70)
print("FINAL DATASET AUDIT SUMMARY")
print("=" * 70)

checks = []

# Case count
checks.append((
    "Total cases = 2238",
    len(metadata) == 2238
))

# Study uniqueness
checks.append((
    "Unique study IDs",
    metadata[STUDY_ID_COL].nunique() == len(metadata)
))

# Patient IDs
checks.append((
    "Patient IDs present",
    metadata[PATIENT_ID_COL].notna().all()
))

# Diagnosis
checks.append((
    "Diagnosis values valid",
    set(metadata[DIAGNOSIS_COL].dropna().unique())
    <= {"PDAC", "non-PDAC"}
))

# Files
checks.append((
    "No missing processed images",
    len(missing_images) == 0
))

checks.append((
    "No missing processed masks",
    len(missing_masks) == 0
))

# Patient diagnosis consistency
checks.append((
    "No patient diagnosis conflicts",
    len(inconsistent_patients) == 0
))

for name, passed in checks:
    print(
        f"{'✓' if passed else '✗'} {name}"
    )

print("\n" + "-" * 70)

if all(passed for _, passed in checks):
    print("✓ FINAL DATASET INTEGRITY CHECK PASSED")
    print("✓ Dataset is ready for patient-level split design.")
else:
    print("✗ DATASET AUDIT REQUIRES INVESTIGATION")
    print("Do NOT create the final train/validation/test split yet.")

FINAL DATASET AUDIT SUMMARY
✓ Total cases = 2238
✓ Unique study IDs
✓ Patient IDs present
✓ Diagnosis values valid
✓ No missing processed images
✓ No missing processed masks
✓ No patient diagnosis conflicts

----------------------------------------------------------------------
✓ FINAL DATASET INTEGRITY CHECK PASSED
✓ Dataset is ready for patient-level split design.
